# Colab runner
Thin launcher only: **no logic lives here** (it is all in `src/` and `scripts/`). Use a **CPU** runtime (Runtime > Change runtime type).
One-time per person: add your GitHub token as a Colab Secret named `GH_TOKEN` (key icon in the left bar, turn on *Notebook access*).
Run the cells top to bottom; each pipeline stage is its own cell so you can run only yours.

In [28]:
# ---------- CONFIG (edit) ----------
GITHUB_REPO  = "manahilcpkhan/tomato-robust-ml"
GIT_USER, GIT_EMAIL = "manahilcpkhan", "manahilkhan74425@gmail.com"
DATA_VARIANT = "Dataset (raw)"        # folder inside the zip used as data_root (raw photos, we make our own split)
MY_BRANCH    = "a/data"                # a/... (Member A), b/... (Member B), c/... (Member C)
# -----------------------------------

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import glob, os, shutil, subprocess, sys
from google.colab import drive, userdata
drive.mount("/content/drive")

def sh(*cmd, check=True):
    return subprocess.run(list(cmd), check=check)

# --- repo
token = userdata.get("GH_TOKEN")
REPO = "/content/repo"
if not os.path.exists(REPO):
    sh("git", "clone", f"https://{token}@github.com/{GITHUB_REPO}.git", REPO)
else:
    sh("git", "-C", REPO, "pull", "--rebase")
os.chdir(REPO)
sh("git", "config", "user.name", GIT_USER); sh("git", "config", "user.email", GIT_EMAIL)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")

# --- dataset: unzip the zip from Drive to LOCAL disk (reading thousands of files straight from Drive is very slow)
zips = glob.glob("/content/drive/MyDrive/**/Tomato_Leaf_Disease_Dataset_Pakistan.zip", recursive=True)
assert zips, "zip not found in My Drive: check the file name / that Drive is mounted"
ZIP = zips[0]
DRIVE_PROJECT = os.path.dirname(ZIP)
DRIVE_BACKUP = os.path.join(DRIVE_PROJECT, "work_backup")
if not os.path.exists("/content/data/.done"):
    os.makedirs("/content/data", exist_ok=True)
    sh("unzip", "-q", "-o", ZIP, "-d", "/content/data")
    open("/content/data/.done", "w").close()
hits = [d for d, _, _ in os.walk("/content/data") if os.path.basename(d) == DATA_VARIANT]
assert hits, f"no folder named {DATA_VARIANT!r} under /content/data"
DATA = hits[0]

# --- paths for the pipeline: big files on local disk, small results inside the repo
os.environ["TOMATO_DATA_ROOT"]   = DATA
os.environ["TOMATO_WORK_DIR"]    = "/content/work"
os.environ["TOMATO_RESULTS_DIR"] = f"{REPO}/results"
print("data_root:", DATA, "| backup:", DRIVE_BACKUP, "| cpus:", os.cpu_count())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data_root: /content/data/Tomato Dataset/Dataset (raw) | backup: /content/drive/MyDrive/ML_Project_Tomato_Leaf_Disease/work_backup | cpus: 2


## Restore previous work from Drive
Colab wipes local disk when a session ends. Run this in every NEW session after the first (it copies `work/` back).

In [14]:
if os.path.exists(DRIVE_BACKUP):
    shutil.copytree(DRIVE_BACKUP, "/content/work", dirs_exist_ok=True)
    print("restored work/ from Drive")
else:
    print("no backup yet (normal on the very first run)")

restored work/ from Drive


## 00 Audit dataset (Member A): class names, counts, near-duplicates
Read the printout. With `--hash` it also clusters near-identical photos (1-3 min) so you can see how many copies exist.

In [15]:
!python scripts/00_audit_dataset.py --hash

Hashing 830 images to cluster near-duplicates (reads every file once) ...
near-duplicate pairs: 55 (9 across different classes = possible label noise). groups 830 -> 785; largest group: 3 images
data_root: /content/data/Tomato Dataset/Dataset (raw)
830 images, 6 classes: ['early_blight', 'healthy', 'late_blight', 'leaf_mold', 'septoria_leaf_spot', 'yellow_leaf_curl_virus']

images per class (after merging near-duplicates: unique photos per class below)
                        images  unique_photos
label                                        
septoria_leaf_spot         195            172
leaf_mold                  186            186
healthy                    137            131
early_blight               119            113
late_blight                113            105
yellow_leaf_curl_virus      80             78
imbalance (largest/smallest class): 2.4x

No train/val/test folders found under this data_root -> official split unavailable (use split_strategy: group).

source photos (group

In [16]:
  # !find "{DATA}" -type f | wc -l

## 01 Prepare data (Member A): grouped train/val/test split + manifest + image cache
Splits by source photo (near-duplicates stay together), 70/15/15 per class, seed from `config.yaml`. Run once, then commit `data/manifest.csv` and `results/grouping_summary.json` via a PR so everyone uses the SAME split.

In [17]:
!python scripts/01_prepare_data.py

Hashing 830 images to cluster near-duplicates (reads every file once) ...
near-duplicate pairs: 55 (9 across different classes = possible label noise). groups 830 -> 785; largest group: 3 images
Wrote /content/repo/data/manifest.csv  (830 images, 785 source photos)
split                   train  val  test
label                                   
early_blight               84   18    17
healthy                    96   21    20
late_blight                80   17    16
leaf_mold                 131   28    27
septoria_leaf_spot        137   29    29
yellow_leaf_curl_virus     56   12    12
Building image cache (830 images -> 256x256) ...
Saved /content/work/images_cache.npy  (0.16 GB)


## 02 EDA, segmentation check, duplicates (Member A)

In [18]:
!python scripts/02_eda_duplicates.py

[counts + original sizes] ...
      width  height
min   150.0   143.0
50%   800.0  1035.5
max  4416.0  4160.0
{'landscape': 472, 'portrait': 358}
[counts + original sizes] done in 0.6s
[sample grid] ...
[sample grid] done in 2.8s
[class brightness stats] ...
                       mean_V       mean_S      
                         mean   std   mean   std
label                                           
early_blight            139.0  17.3   98.9  36.7
healthy                 147.0  13.8   90.2  31.4
late_blight             132.6  21.9  110.0  36.2
leaf_mold               150.3  17.3  125.9  28.4
septoria_leaf_spot      136.4  20.7  116.8  31.5
yellow_leaf_curl_virus  140.9  17.2  102.9  42.3
[class brightness stats] done in 1.6s
[segmentation check] ...
{'method': 'hsv', 'n_checked': 830, 'mask_frac_mean': 0.8297835936029273, 'mask_frac_median': 0.8914260864257812, 'frac_mask_above_0.95': 0.3060240963855422, 'frac_fallback_whole_image': 0.06385542168674699}
[segmentation check] done in 

## 03 Features (Member B)
Benchmark first, then the full extraction.

In [19]:
!python scripts/03_extract_features.py --benchmark 100

105 features: color=47, texture=38, shape=20
100 images in 11.4s -> 114 ms/img (wall). Estimated full dataset: 1.6 min. NaNs: 0


In [20]:
!python scripts/03_extract_features.py

105 features: color=47, texture=38, shape=20
train: (584, 105) in 50s
val: (125, 105) in 12s
test: (121, 105) in 10s


## 04 Baseline (RF / SVM / LR)  ·  05 Ablation (Member B)

In [21]:
!python scripts/04_baseline.py

Leakage flags found: 3/121 test images have a near-duplicate or same source photo in train/val
[tune rf on validation] ...
[tune rf on validation] done in 38.4s
  best rf: {'max_depth': None, 'max_features': 0.3, 'min_samples_leaf': 1}  val macro-F1=0.5821
  rf: test acc=0.6198 macro-F1=0.6098 [0.507, 0.691]
[tune svm on validation] ...
[tune svm on validation] done in 0.5s
  best svm: {'C': 10, 'gamma': 0.001}  val macro-F1=0.6175
  svm: test acc=0.6281 macro-F1=0.6058 [0.505, 0.691]
[tune lr on validation] ...
[tune lr on validation] done in 0.4s
  best lr: {'C': 0.1}  val macro-F1=0.6258
  lr: test acc=0.6033 macro-F1=0.5924 [0.501, 0.675]
[RF learning curve] ...
[RF learning curve] done in 15.0s


In [22]:
!python scripts/05_ablation.py

[rf / color (47 feats)] ...
[rf / color (47 feats)] done in 4.0s
[svm / color (47 feats)] ...
[svm / color (47 feats)] done in 0.0s
[lr / color (47 feats)] ...
[lr / color (47 feats)] done in 0.0s
[rf / texture (38 feats)] ...
[rf / texture (38 feats)] done in 2.3s
[svm / texture (38 feats)] ...
[svm / texture (38 feats)] done in 0.0s
[lr / texture (38 feats)] ...
[lr / texture (38 feats)] done in 0.0s
[rf / shape (20 feats)] ...
[rf / shape (20 feats)] done in 1.7s
[svm / shape (20 feats)] ...
[svm / shape (20 feats)] done in 0.1s
[lr / shape (20 feats)] ...
[lr / shape (20 feats)] done in 0.1s
[rf / color+texture (85 feats)] ...
[rf / color+texture (85 feats)] done in 4.5s
[svm / color+texture (85 feats)] ...
[svm / color+texture (85 feats)] done in 0.0s
[lr / color+texture (85 feats)] ...
[lr / color+texture (85 feats)] done in 0.0s
[rf / color+shape (67 feats)] ...
[rf / color+shape (67 feats)] done in 3.2s
[svm / color+shape (67 feats)] ...
[svm / color+shape (67 feats)] done in 0

## 06 Robustness  ·  07 Importance stability  ·  08 Error analysis  ·  09 Figures (Member C)

In [23]:
!python scripts/06_robustness.py

[fit models on clean train+val] ...
[fit models on clean train+val] done in 26.9s
[brightness_0.5] extracted (121, 105) in 10s
[brightness_0.7] extracted (121, 105) in 10s
[brightness_1.3] extracted (121, 105) in 11s
[brightness_1.6] extracted (121, 105) in 8s
[contrast_0.5] extracted (121, 105) in 10s
[contrast_0.7] extracted (121, 105) in 10s
[contrast_1.5] extracted (121, 105) in 8s
[contrast_2.0] extracted (121, 105) in 11s
[shadow_0.3] extracted (121, 105) in 12s
[shadow_0.5] extracted (121, 105) in 10s
[shadow_0.7] extracted (121, 105) in 11s
[background_gray] extracted (121, 105) in 11s
[background_noise] extracted (121, 105) in 9s
[background_soil] extracted (121, 105) in 11s
[background_grass] extracted (121, 105) in 12s
       condition  accuracy  macro_f1
           clean    0.6198    0.6098
  brightness_0.5    0.5455    0.5170
  brightness_0.7    0.5950    0.5813
  brightness_1.3    0.5289    0.4991
  brightness_1.6    0.4132    0.3633
    contrast_0.5    0.3388    0.3365
 

In [24]:
!python scripts/07_importance_stability.py

[CV fold 1/5] ...
[CV fold 1/5] done in 56.1s
[CV fold 2/5] ...
[CV fold 2/5] done in 55.5s
[CV fold 3/5] ...
[CV fold 3/5] done in 58.1s
[CV fold 4/5] ...
[CV fold 4/5] done in 59.1s
[CV fold 5/5] ...
[CV fold 5/5] done in 56.7s
[permutation importance on brightness_0.5] ...
[permutation importance on brightness_0.5] done in 29.2s
[permutation importance on brightness_0.7] ...
[permutation importance on brightness_0.7] done in 30.6s
[permutation importance on brightness_1.3] ...
[permutation importance on brightness_1.3] done in 29.8s
[permutation importance on brightness_1.6] ...
[permutation importance on brightness_1.6] done in 29.2s
[permutation importance on contrast_0.5] ...
[permutation importance on contrast_0.5] done in 29.7s
[permutation importance on contrast_0.7] ...
[permutation importance on contrast_0.7] done in 32.5s
[permutation importance on contrast_1.5] ...
[permutation importance on contrast_1.5] done in 30.6s
[permutation importance on contrast_2.0] ...
[permutat

In [25]:
!python scripts/08_error_analysis.py

                  true              predicted  count  rate_of_true_class
          early_blight     septoria_leaf_spot      7            0.411765
    septoria_leaf_spot              leaf_mold      5            0.172414
             leaf_mold     septoria_leaf_spot      5            0.185185
yellow_leaf_curl_virus     septoria_leaf_spot      4            0.333333
           late_blight           early_blight      3            0.187500
               healthy yellow_leaf_curl_virus      3            0.150000
               healthy            late_blight      3            0.150000
    septoria_leaf_spot            late_blight      3            0.103448
Worst condition for RF/all: contrast_2.0
                 class  share_pred_clean  share_pred_contrast_2.0
          early_blight             0.074                    0.058
           late_blight             0.132                    0.017
    septoria_leaf_spot             0.306                    0.256
             leaf_mold             0.2

In [26]:
!python scripts/09_figures.py

figures written to /content/repo/results


## Save: back up `work/` to Drive and push small results to your branch
Then open a pull request on GitHub. Never push to `main` from the notebook.

In [27]:
os.makedirs(DRIVE_BACKUP, exist_ok=True)
if os.path.exists("/content/work"):
    shutil.copytree("/content/work", DRIVE_BACKUP, dirs_exist_ok=True)
sh("git", "checkout", "-B", MY_BRANCH)
sh("git", "add", "results", "data/manifest.csv", "config.yaml", check=False)
sh("git", "commit", "-m", "results: update from notebook run", check=False)
sh("git", "push", "-u", "origin", MY_BRANCH)

CompletedProcess(args=['git', 'push', '-u', 'origin', 'a/data'], returncode=0)